# Phase 3: Model Development & Tuning
This notebook trains baseline and advanced ensemble models and performs hyperparameter tuning.


## Step 1: Load Features and Split
We load the engineered features, partition the data into train, val, and test, and scale them.


In [1]:
import pandas as pd
import numpy as np
from src.data_pipeline.preprocessing import split_data, scale_features
from src.models.baseline import train_logistic_regression, train_decision_tree
from src.models.advanced import train_random_forest, train_gradient_boosting, tune_hyperparameters, train_multilabel_classifier
from src.models.ensemble import train_voting_classifier, train_stacking_classifier, compare_ensemble_models

df = pd.read_csv('data/features/engineered_features.csv')
X = df.drop(columns=['Machine failure', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF'])
y = df['Machine failure']

X_train, X_val, X_test, y_train, y_val, y_test = split_data(
    pd.concat([X, y], axis=1), 'Machine failure'
)
X_train_s, X_val_s, X_test_s, scaler = scale_features(X_train, X_val, X_test)
print('Data scaled successfully.')


Data scaled successfully.


## Step 2: Hyperparameter Tuning
We perform a grid/random search over hyperparameters for RF and GB.


In [2]:
print('Tuning Random Forest...')
rf_tuned = tune_hyperparameters('random_forest', X_train_s, y_train, cv=3)
print('RF Best params:', rf_tuned['best_params'])
print('RF Best CV F1:', rf_tuned['best_score'])

print('\nTuning Gradient Boosting...')
gb_tuned = tune_hyperparameters('gradient_boosting', X_train_s, y_train, cv=3)
print('GB Best params:', gb_tuned['best_params'])
print('GB Best CV F1:', gb_tuned['best_score'])


Tuning Random Forest...


RF Best params: {'n_estimators': 300, 'min_samples_split': 5, 'min_samples_leaf': 2, 'max_depth': 15}
RF Best CV F1: 0.8336

Tuning Gradient Boosting...


GB Best params: {'subsample': 0.7, 'n_estimators': 200, 'max_depth': 4, 'learning_rate': 0.01}
GB Best CV F1: 0.9146


## Step 3: Ensemble & Stacking
We train soft voting and stacking meta-ensembles and compare validation performance.


In [3]:
models = {
    'voting': train_voting_classifier(X_train_s, y_train),
    'stacking': train_stacking_classifier(X_train_s, y_train),
    'random_forest_tuned': rf_tuned['best_estimator'],
    'gradient_boosting_tuned': gb_tuned['best_estimator']
}
comparison_df = compare_ensemble_models(models, X_val_s, y_val)
print(comparison_df)


                model_name  accuracy  precision  recall      f1  roc_auc
0                   voting     0.992     0.9643  0.7941  0.8710   0.9770
1                 stacking     0.991     0.9630  0.7647  0.8525   0.9836
2      random_forest_tuned     0.990     0.9286  0.7647  0.8387   0.9925
3  gradient_boosting_tuned     0.960     0.4545  0.8824  0.6000   0.9931


## Step 4: Multi-Label Failure Mode Classification
We train a MultiOutput Classifier to identify specific failure modes.


In [4]:
y_multilabel = df[['TWF', 'HDF', 'PWF', 'OSF', 'RNF']]
y_multi_train = y_multilabel.loc[X_train.index].values
y_multi_val = y_multilabel.loc[X_val.index].values
y_multi_test = y_multilabel.loc[X_test.index].values

ml_classifier = train_multilabel_classifier(X_train_s, y_multi_train)
print('Multi-label classifier trained.')


Multi-label classifier trained.
